# House Price Prediction
## Notebook 1 — Project Overview and Data Inspection
**Author: Tinashe Clifford Makandise**

### 1. Project purpose
Estimate historical residential sale prices in Ames, Iowa, from property characteristics and build an interactive Python/Streamlit application. The target is `SalePrice` in USD. This educational model will not estimate current market values or prices in other countries.

### Workflow
1. Overview, data inspection and holdout creation (this notebook).
2. Exploratory data analysis using training data.
3. Preprocessing and feature engineering.
4. Model comparison with cross-validation.
5. Final holdout evaluation and export.

Model selection will use training data only. MAE is the primary metric; RMSE and R² provide additional context. A median-price baseline will be compared with Linear Regression, Ridge and Random Forest. No model is trained in this notebook.

## 2. Setup and source
Place this notebook in `house-price-prediction/notebooks/`. Download [AmesHousing.txt](https://jse.amstat.org/v19n3/decock/AmesHousing.txt) into `house-price-prediction/data/raw/`. The code also works when Jupyter is launched from the project root.

Install dependencies once in a notebook cell with `%pip install pandas numpy scikit-learn`. Restart the kernel if requested.

**Source:** Dean De Cock (2011), *Ames, Iowa: Alternative to the Boston Housing Data as an End of Semester Regression Project*, Journal of Statistics Education, 19(3). [Original paper](https://jse.amstat.org/v19n3/decock.pdf) · [Data dictionary](https://jse.amstat.org/v19n3/decock/DataDocumentation.txt).

The original file contains 2,930 sales from 2006–2010 and 82 columns including identifiers and the target. Preserve it unchanged. Retain attribution and check redistribution permissions before publishing raw data.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string() if hasattr(value, "to_string") else value)

RANDOM_STATE = 42
TEST_SIZE = 0.20
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "AmesHousing.txt"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
if not RAW_PATH.is_file():
    raise FileNotFoundError(f"Download AmesHousing.txt into: {RAW_PATH.parent}")
print("Python:", platform.python_version())
print("pandas:", pd.__version__, "scikit-learn:", sklearn.__version__)

Python: 3.12.14
pandas: 2.2.3 scikit-learn: 1.8.0


## 3. Load and validate the source
The file is tab-separated. Default parsing of `NA` would collapse an explicit absence code and an empty field into the same missing value. We preserve literal `NA` strings here and parse blank fields as missing; later preprocessing will interpret each feature using its documentation.

Only structural and integrity checks are performed on the full source before splitting. Target distributions and relationships are explored later using training data.

In [2]:
data = pd.read_csv(RAW_PATH, sep="\t", keep_default_na=False, na_values=[""])
assert data.shape == (2930, 82), f"Unexpected source shape: {data.shape}"
assert data.columns.is_unique
required = ["PID", "SalePrice", "Gr Liv Area", "Bedroom AbvGr", "Full Bath",
            "Half Bath", "Neighborhood", "Year Built", "Yr Sold", "Garage Cars"]
assert set(required).issubset(data.columns)
assert pd.api.types.is_numeric_dtype(data["SalePrice"])
assert data["SalePrice"].notna().all() and data["SalePrice"].gt(0).all()
assert data["PID"].notna().all() and data["PID"].is_unique, "Repeated properties need grouped splitting."
assert not data.duplicated().any(), "Investigate duplicate rows before splitting."
source_hash = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()
print(f"Validated {len(data):,} rows and {data.shape[1]} columns.")
print("Source SHA-256:", source_hash)

Validated 2,930 rows and 82 columns.
Source SHA-256: 6cfe6cb525ba437de428653a1040e2aed7d696640bf75203786a6d7a0e67cfcc


## 4. Reserve the test set
A reproducible random 80/20 split measures performance on held-out properties from the same historical population. It does not demonstrate forecasting future prices or transferring to a different city. PID uniqueness is verified to prevent the same property entering both sets.

The untouched test file contains the target for eventual evaluation. Do not use it for exploration, preprocessing decisions, model selection or tuning.

In [3]:
train_data, test_data = train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_data = train_data.sort_values("Order").copy()
test_data = test_data.sort_values("Order").copy()
assert set(train_data["PID"]).isdisjoint(set(test_data["PID"]))
assert len(train_data) + len(test_data) == len(data)
print(f"Training rows: {len(train_data):,}; reserved test rows: {len(test_data):,}")
del data

Training rows: 2,344; reserved test rows: 586


## 5. Inspect the training data
All remaining data inspection uses `train_data`. `info()` describes types and non-null counts; the preview helps verify import alignment. `Order` and `PID` are identifiers and will not be predictors. `MS SubClass` is numerically coded but describes a category.

In [4]:
display(train_data.head())
train_data.info()
display(train_data.describe().T)

   Order        PID  MS SubClass MS Zoning  Lot Frontage  Lot Area Street Alley Lot Shape Land Contour Utilities Lot Config Land Slope Neighborhood Condition 1 Condition 2 Bldg Type House Style  Overall Qual  Overall Cond  Year Built  Year Remod/Add Roof Style Roof Matl Exterior 1st Exterior 2nd Mas Vnr Type  Mas Vnr Area Exter Qual Exter Cond Foundation Bsmt Qual Bsmt Cond Bsmt Exposure BsmtFin Type 1  BsmtFin SF 1 BsmtFin Type 2  BsmtFin SF 2  Bsmt Unf SF  Total Bsmt SF Heating Heating QC Central Air Electrical  1st Flr SF  2nd Flr SF  Low Qual Fin SF  Gr Liv Area  Bsmt Full Bath  Bsmt Half Bath  Full Bath  Half Bath  Bedroom AbvGr  Kitchen AbvGr Kitchen Qual  TotRms AbvGrd Functional  Fireplaces Fireplace Qu Garage Type  Garage Yr Blt Garage Finish  Garage Cars  Garage Area Garage Qual Garage Cond Paved Drive  Wood Deck SF  Open Porch SF  Enclosed Porch  3Ssn Porch  Screen Porch  Pool Area Pool QC  Fence Misc Feature  Misc Val  Mo Sold  Yr Sold Sale Type Sale Condition  SalePrice
0 

### 5.1 Missing values and absence codes
A blank field means missing information under our parser. A literal `NA` may mean a feature is absent (such as no alley, basement or garage), depending on the data dictionary. Do not globally replace it or impute it without checking the feature definition.

In [5]:
quality = pd.DataFrame({
    "dtype": train_data.dtypes.astype(str),
    "missing_count": train_data.isna().sum(),
    "missing_percent": train_data.isna().mean().mul(100),
    "literal_NA_count": train_data.eq("NA").sum(),
    "unique_non_null": train_data.nunique(dropna=True),
})
display(quality.sort_values(["missing_count", "literal_NA_count"], ascending=False).head(25))

                  dtype  missing_count  missing_percent  literal_NA_count  unique_non_null
Lot Frontage    float64            393        16.766212                 0              120
Garage Yr Blt   float64            122         5.204778                 0              102
Mas Vnr Type     object             19         0.810580                 0                5
Mas Vnr Area    float64             19         0.810580                 0              391
Bsmt Exposure    object              3         0.127986                60                5
Garage Finish    object              2         0.085324               120                4
BsmtFin Type 2   object              2         0.085324                60                7
Garage Qual      object              1         0.042662               121                6
Garage Cond      object              1         0.042662               121                6
Bsmt Qual        object              1         0.042662                60                6

### 5.2 Candidate inputs and integrity flags
The initial inputs align with the proposed app. Area means above-ground living area, not land size. Bedrooms refer to above-ground bedrooms. Keep full and half bathrooms distinct initially; basement bathroom features may be added later. Age will be computed **at sale**, not using today's year. Garage spaces are not total parking availability. Quality and condition may improve the model but will be assessed using training data.

Flags below indicate records to investigate, not automatic deletion rules. No outliers are removed here.

In [6]:
feature_spec = pd.DataFrame([
    ("Gr Liv Area", "Above-ground living area", "square feet"),
    ("Bedroom AbvGr", "Above-ground bedrooms", "count"),
    ("Full Bath", "Above-ground full bathrooms", "count"),
    ("Half Bath", "Above-ground half bathrooms", "count"),
    ("Neighborhood", "Location within Ames", "category"),
    ("Year Built + Yr Sold", "Age at sale = Yr Sold - Year Built", "years"),
    ("Garage Cars", "Garage capacity", "cars"),
    ("Overall Qual", "Overall material/finish quality", "ordinal rating"),
    ("Overall Cond", "Overall condition", "ordinal rating"),
], columns=["source_columns", "meaning", "unit"])
display(feature_spec)
checks = pd.Series({
    "nonpositive_living_area": train_data["Gr Liv Area"].le(0).sum(),
    "negative_bedrooms": train_data["Bedroom AbvGr"].lt(0).sum(),
    "negative_garage_capacity": train_data["Garage Cars"].lt(0).sum(),
    "built_after_sale_year": train_data["Year Built"].gt(train_data["Yr Sold"]).sum(),
    "living_area_over_4000_sqft_review": train_data["Gr Liv Area"].gt(4000).sum(),
}, name="training_records_flagged")
display(checks.to_frame())

         source_columns                             meaning            unit
0           Gr Liv Area            Above-ground living area     square feet
1         Bedroom AbvGr               Above-ground bedrooms           count
2             Full Bath         Above-ground full bathrooms           count
3             Half Bath         Above-ground half bathrooms           count
4          Neighborhood                Location within Ames        category
5  Year Built + Yr Sold  Age at sale = Yr Sold - Year Built           years
6           Garage Cars                     Garage capacity            cars
7          Overall Qual     Overall material/finish quality  ordinal rating
8          Overall Cond                   Overall condition  ordinal rating
                                   training_records_flagged
nonpositive_living_area                                   0
negative_bedrooms                                         0
negative_garage_capacity                                  0


## 6. Export the handover files
CSV exports retain original column names. Later notebooks must load them with the same missing-value settings to preserve `NA` absence codes. Outputs are overwritten deterministically when rerun with this source and split seed. These partitions are raw splits, not cleaned or encoded data.

The manifest records provenance, split membership and package versions for reproducibility. No imputer, encoder or scaler has been fitted.

In [7]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
train_data.to_csv(PROCESSED_DIR / "train.csv", index=False)
test_data.to_csv(PROCESSED_DIR / "test.csv", index=False)
quality.to_csv(REPORTS_DIR / "training_data_quality.csv", index_label="column")
feature_spec.to_csv(REPORTS_DIR / "candidate_features.csv", index=False)
checks.to_csv(REPORTS_DIR / "training_integrity_flags.csv", index_label="check")
manifest = {
    "source_url": "https://jse.amstat.org/v19n3/decock/AmesHousing.txt",
    "source_sha256": source_hash,
    "target": "SalePrice", "currency": "USD", "sale_years": "2006–2010",
    "random_state": RANDOM_STATE, "test_fraction": TEST_SIZE,
    "train_rows": len(train_data), "test_rows": len(test_data),
    "train_pids": train_data["PID"].astype(int).tolist(),
    "test_pids": test_data["PID"].astype(int).tolist(),
    "csv_read_settings": {"keep_default_na": False, "na_values": [""]},
    "versions": {"python": platform.python_version(), "pandas": pd.__version__,
                 "numpy": np.__version__, "scikit_learn": sklearn.__version__},
}
(REPORTS_DIR / "data_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Saved training/test partitions, quality reports, candidate features and provenance manifest.")

Saved training/test partitions, quality reports, candidate features and provenance manifest.


## 7. Findings and next stage
The original source has passed structural checks and a separate test set has been reserved. The generated tables document training-data types, missingness, absence codes and integrity flags; they are evidence for subsequent decisions rather than a completed cleaning plan.

**Decisions:** preserve the source, exclude identifiers from modelling, retain explicit absence codes, use training data for exploration and cross-validation, and evaluate the final selected model once on the holdout. Do not infer causal effects from later correlations or feature importance.

**Next notebook:** explore the training sale-price distribution and relationships with living area, neighbourhood, quality, age and garage capacity. Investigate flagged observations and document justified cleaning decisions before building preprocessing pipelines.

To reload training data in Notebook 2:
```python
train_data = pd.read_csv(PROJECT_ROOT / "data/processed/train.csv",
                         keep_default_na=False, na_values=[""])
```